# **Análisis de una Base de Datos de Libros mediante SQL**

# Contexto del Proyecto

El coronavirus tomó al mundo entero por sorpresa, cambiando la rutina diaria de todos y todas. Los habitantes de las ciudades ya no pasaban su tiempo libre fuera, yendo a cafés y centros comerciales; sino que más gente se quedaba en casa, leyendo libros. Eso atrajo la atención de las startups (empresas emergentes) que se apresuraron a desarrollar nuevas aplicaciones para los amantes de los libros.

Te han dado una base de datos de uno de los servicios que compiten en este mercado. Contiene datos sobre libros, editoriales, autores y calificaciones de clientes y reseñas de libros. Esta información se utilizará para generar una propuesta de valor para un nuevo producto.

### Requerimientos de la Consigna

- Encuentra el número de libros publicados después del 1 de enero de 2000.
- Encuentra el número de reseñas de usuarios y la calificación promedio para cada libro.
- Identifica la editorial que ha publicado el mayor número de libros con más de 50 páginas (esto te ayudará a excluir folletos y publicaciones similares de tu análisis).
- Identifica al autor que tiene la más alta calificación promedio del libro: mira solo los libros con al menos 50 calificaciones.
- Encuentra el número promedio de reseñas de texto entre los usuarios que calificaron más de 50 libros.

# Descripción de los Datos

La base de datos contiene información relacionada con libros, autores, editoriales, calificaciones de usuarios y reseñas.

## Tablas disponibles

### `books`

Contiene información sobre los libros:

- `book_id`: identificación del libro.
- `author_id`: identificación del autor o autora.
- `title`: título.
- `num_pages`: número de páginas.
- `publication_date`: fecha de publicación.
- `publisher_id`: identificación de la editorial.

### `authors`

Contiene información sobre los autores:

- `author_id`: identificación del autor o autora.
- `author`: nombre del autor o autora.

### `publishers`

Contiene información sobre las editoriales:

- `publisher_id`: identificación de la editorial.
- `publisher`: nombre de la editorial.

### `ratings`

Contiene las calificaciones realizadas por los usuarios:

- `rating_id`: identificación de la calificación.
- `book_id`: identificación del libro.
- `username`: usuario que realizó la calificación.
- `rating`: calificación otorgada.

### `reviews`

Contiene las reseñas realizadas por los clientes:

- `review_id`: identificación de la reseña.
- `book_id`: identificación del libro.
- `username`: usuario que realizó la reseña.
- `text`: contenido de la reseña.

## Relaciones entre las tablas
!["Diagrama de Datos"](Diagrama_base_datos.png)

# Introducción General del Proyecto

El objetivo del proyecto es analizar una base de datos de una plataforma relacionada con libros para obtener información que pueda contribuir al desarrollo de una propuesta de valor para un nuevo producto.

El análisis se realizará mediante SQL, utilizando las relaciones existentes entre libros, autores, editoriales, calificaciones y reseñas. Primero se validará la estructura y el estado inicial de las tablas; posteriormente se realizarán las consultas necesarias para responder las cinco preguntas planteadas en la encomienda.

El resultado esperado es obtener indicadores sobre la evolución de las publicaciones, el comportamiento de las calificaciones y reseñas, el desempeño de las editoriales y autores, y la actividad de los usuarios, utilizando exclusivamente SQL para la obtención de los resultados.

# 1. Inicialización y Carga de los Datos

El objetivo de esta parte es diagnosticar el estado inicial de los datasets, verificando su estructura, dimensiones, tipos de datos, valores ausentes, duplicados y otras condiciones relevantes, con el fin de identificar las necesidades de transformación y preparación que deberán abordarse en la Parte de Preprocesamiento.

## 1.1 Configuración del entorno y conexión con la base de datos

- Importación de las librerías necesarias para la conexión y manipulación de los resultados.
- Configuración de la conexión con la base de datos mediante SQLAlchemy.
- Establecimiento de la conexión a través de la variable `engine`.
- Validación de que la conexión con la base de datos se haya establecido correctamente.

### Importación de librerías

In [1]:
# Importación de librerías
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from IPython.display import display as d, Markdown as m

load_dotenv()

True

### Creación y Configuración de la Conexión a la Base De Datos

- La conexión se establece con una base de datos PostgreSQL alojada en un servidor remoto. Se utiliza SQLAlchemy para gestionar la conexión y SSL para proteger la comunicación con el servidor.

In [2]:
# Configuración de la conexión
db_config = {
    'user': os.getenv('DB_USER'),
    'pwd': os.getenv('DB_PASSWORD'),
    'host': os.getenv('DB_HOST'),
    'port': os.getenv('DB_PORT'),
    'db': os.getenv('DB_NAME')
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(
    connection_string,
    connect_args={'sslmode': 'require'}
)

### Validación de la conexión

In [3]:
query = "SELECT 1;"

connection_test = pd.read_sql(query, engine)

connection_test

,?column?
0,1


### Acceso a las tablas

In [4]:
# Obtener todas las tablas disponibles en la base de datos

query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

all_tables = pd.read_sql(query, engine)

all_tables

,table_name
0,advertisment_costs
1,authors
2,books
3,check_avg
4,orders
5,publishers
6,ratings
7,reviews
8,visits


In [5]:
# Tablas que utilizaremos en este proyecto


tables = [
    'books',
    'authors',
    'publishers',
    'ratings',
    'reviews'
]

tables

['books', 'authors', 'publishers', 'ratings', 'reviews']

### Impresión de las Primeras Filas

## 1.2 Carga y reconocimiento de las tablas

- Acceso a las tablas `books`, `authors`, `publishers`, `ratings` y `reviews`.
- Impresión de las primeras filas de cada tabla para conocer su contenido.
- Revisión de dimensiones y estructura inicial de cada tabla, tipos de datos, validación de valores nulos, validación de registros duplicados e Identificación de cualquier condición relevante que deba considerarse antes del análisis.

In [6]:
# Mostrar las primeras 5 filas de las tablas seleccionadas

for table in tables:
    query = f"""
    SELECT *
    FROM {table}
    LIMIT 5;
    """

    print(f"\n{'=' * 60}")
    d(m(f"## Tabla: {table}"))
    print()

    display(pd.read_sql(query, engine))

## Tabla: books

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


## Tabla: authors

,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


## Tabla: publishers

,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


## Tabla: ratings

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


## Tabla: reviews

,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


### Dimensiones, Estructura inicial, Tipos de Datos, Valores Ausentes y Duplicados de cada Tabla

In [7]:
# Bucle para revisar las tablas seleccionadas
for table in tables:

    # Obtener estructura de la tabla
    structure_query = f"""
    SELECT
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name = '{table}'
    ORDER BY ordinal_position;
    """

    structure = pd.read_sql(structure_query, engine)

    columns = structure['column_name'].tolist()

    # Total de registros
    records_query = f"""
    SELECT COUNT(*) AS total_registros
    FROM {table};
    """

    total_records = pd.read_sql(records_query, engine)

    # Filas completas duplicadas
    duplicate_query = f"""
    SELECT COALESCE(SUM(cantidad - 1), 0) AS filas_duplicadas
    FROM (
        SELECT COUNT(*) AS cantidad
        FROM {table}
        GROUP BY {', '.join(f'"{column}"' for column in columns)}
        HAVING COUNT(*) > 1
    ) AS duplicates;
    """

    duplicates = pd.read_sql(duplicate_query, engine)

    # Valores ausentes por columna
    null_query = f"""
    SELECT
        {', '.join(
        [f'COUNT(*) FILTER (WHERE "{column}" IS NULL) AS "{column}"'
         for column in columns]
    )}
    FROM {table};
    """

    nulls = pd.read_sql(null_query, engine)

    # Integrar estructura y valores ausentes
    structure['valores_ausentes'] = [
        nulls[column].iloc[0]
        for column in columns
    ]

    # Renombrar columnas para la presentación
    structure = structure.rename(columns={
        'column_name': 'Nombre de la columna',
        'data_type': 'Tipo de Datos',
        'valores_ausentes': 'Valores Ausentes'
    })

    # Mostrar resultados
    d(m(f"### Tabla: `{table}`"))

    d(m(
        f"**Total de registros:** "
        f"{total_records['total_registros'].iloc[0]}"
    ))

    d(m(
        f"**Filas completas duplicadas:** "
        f"{duplicates['filas_duplicadas'].iloc[0]}"
    ))

    d(m("**Estructura:**"))
    d(structure)

### Tabla: `books`

**Total de registros:** 1000

**Filas completas duplicadas:** 0.0

**Estructura:**

,Nombre de la columna,Tipo de Datos,Valores Ausentes
0,book_id,integer,0
1,author_id,integer,0
2,title,text,0
3,num_pages,integer,0
4,publication_date,date,0
5,publisher_id,integer,0


...

### Tabla: `reviews`

**Total de registros:** 2793

**Filas completas duplicadas:** 0.0

**Estructura:**

,Nombre de la columna,Tipo de Datos,Valores Ausentes
0,review_id,integer,0
1,book_id,integer,0
2,username,text,0
3,text,text,0


<div style="border-left: 10px solid #616161;padding: 20px;background: #D6D6D6;border-radius: 8px;color: #000000;word-wrap: break-word;overflow-wrap: break-word;white-space: normal;max-width: 98%;box-sizing: border-box;font-size: 15px;line-height: 1.6;text-align: justify;">

## Conclusiones Parte 1. Inicialización y Carga de los Datos

- El diagnóstico inicial permitió verificar la estructura y el estado de las cinco tablas seleccionadas para el proyecto. No se identificaron valores ausentes ni filas completas duplicadas en ninguna de las tablas, y los tipos de datos observados son consistentes con la naturaleza de las variables descritas en la base de datos.

- Con base en estas validaciones, no se identifican correcciones de calidad de datos que deban aplicarse antes de continuar con el análisis. La información se encuentra disponible para avanzar a la siguiente etapa del proyecto.

</div>

# 2. Preprocesamiento de los Datos

El objetivo de esta parte es preparar los datos para el análisis mediante la aplicación de las transformaciones y correcciones justificadas por el diagnóstico inicial, garantizando que la información quede estructurada, consistente y en condiciones adecuadas para desarrollar posteriormente el EDA.

<div style="border-left: 10px solid #616161;padding: 20px;background: #D6D6D6;border-radius: 8px;color: #000000;word-wrap: break-word;overflow-wrap: break-word;white-space: normal;max-width: 98%;box-sizing: border-box;font-size: 15px;line-height: 1.6;text-align: justify;">

## Conclusiones Parte 2. Preprocesamiento de los Datos
De acuerdo con el diagnóstico realizado en la Parte 1, no se identificaron valores ausentes, filas completas duplicadas ni inconsistencias en los tipos de datos de las tablas seleccionadas. Por lo tanto, no se requieren transformaciones ni correcciones de los datos antes de continuar con el análisis exploratorio.

</div>


# 3. Análisis Exploratorio de Datos (EDA)

El objetivo de esta parte es responder mediante consultas SQL las preguntas planteadas en la encomienda y transformar los datos relacionales disponibles en indicadores útiles para la evaluación del mercado de libros. El análisis seguirá un flujo progresivo: primero se examinará la evolución de las publicaciones, después el comportamiento de calificaciones y reseñas, posteriormente el desempeño de editoriales y autores, y finalmente la actividad de los usuarios.

## 3.1 Libros publicados después del 1 de enero de 2000
Determinar la cantidad de libros publicados después del **1 de enero de 2000**.
- Filtrar los libros según `publication_date`.
- Calcular el número total de libros que cumplen el criterio.
- Obtener el resultado e interpretar los resultados.

In [8]:
# Libros publicados después del 1 de enero de 2000

query = """
SELECT COUNT(DISTINCT book_id) AS "Total de Libros"
FROM books
WHERE publication_date > '2000-01-01';
"""

result1 = pd.read_sql(query, engine)

# Mostrar
d(result1)

,Total de Libros
0,819


<div style="border-left: 10px solid #616161;padding: 20px;background: #D6D6D6;border-radius: 8px;color: #000000;word-wrap: break-word;overflow-wrap: break-word;white-space: normal;max-width: 98%;box-sizing: border-box;font-size: 15px;line-height: 1.6;text-align: justify;">

## Conclusiones Parte 3.1 Libros publicados después del 1 de enero de 2000

- De los 1,000 libros registrados en la base de datos, **819 fueron publicados después del 1 de enero de 2000**, lo que representa el **81.9 % del catálogo analizado**. Esto indica que la mayor parte de las publicaciones disponibles corresponde al periodo posterior al año 2000.

</div>

In [9]:
# Total de Libros publicados

query = """
SELECT COUNT(DISTINCT book_id) AS "Total de libros"
FROM books;
"""
result = pd.read_sql(query, engine)

# Mostrar
d(result)

,Total de libros
0,1000


## 3.2 Calificaciones y reseñas por libro
Determinar el número de reseñas de usuarios y la calificación promedio para cada libro.
- Relacionar las tablas `books`, `ratings` y `reviews`.
- Calcular el número de reseñas de usuarios para cada libro.
- Calcular la calificación promedio de cada libro.
- Agrupar los resultados por libro.
- Considerar los libros sin reseñas o calificaciones cuando corresponda.
- Obtener el resultado e interpretar las diferencias observadas entre los libros.

In [10]:
# Número de reseñas y calificación promedio por libro
query = """
SELECT
    b.book_id,
    b.title,
    COUNT(DISTINCT r.review_id) AS "Número de Reseñas por Libro",
    AVG(rt.rating) AS "Calificación Promedio"
FROM books AS b
LEFT JOIN ratings AS rt
    ON b.book_id = rt.book_id
LEFT JOIN reviews AS r
    ON b.book_id = r.book_id
GROUP BY
    b.book_id,
    b.title
ORDER BY
    b.book_id;
"""

result2 = pd.read_sql(query, engine)

# Orden descendente por "Número de Reseñas por Libro"
result2 = result2.sort_values(
    by="Número de Reseñas por Libro", ascending=False).reset_index(drop=True)

# Mostrar Resultados

# Libros sin reseñas
reviews_0 = result2[result2['Número de Reseñas por Libro'] == 0]
d(m(f"### **Libros sin reseñas: {reviews_0.shape[0]}**"))

# Tabla obtenida del Sql en orden descendente por "Número de Reseñas por Libro"
d(m("### **Tabla: Número de Reseñas y Calificación Promedio por Libro**"))
d(result2.head().style.hide(axis="index"))

# Describe()
d(m("""### **Descripción estadística de Número de Reseñas y calificación Promedio por Libro**"""))
d(m("""El conteo no representa el número de reseñas en total, representa el número de libros en la base de datos"""))
result2[["Número de Reseñas por Libro", "Calificación Promedio"]].describe()

### **Libros sin reseñas: 6**

### **Tabla: Número de Reseñas y Calificación Promedio por Libro**

book_id,title,Número de Reseñas por Libro,Calificación Promedio
948,Twilight (Twilight #1),7,3.662500
963,Water for Elephants,6,3.977273
497,Outlander (Outlander #1),6,4.125000
627,The Alchemist,6,3.789474
696,The Da Vinci Code (Robert Langdon #2),6,3.830508


### **Descripción estadística de Número de Reseñas y calificación Promedio por Libro**

El conteo no representa el número de reseñas en total, representa el número de libros en la base de datos

,Número de Reseñas por Libro,Calificación Promedio
count,1000.000000,1000.000000
mean,2.793000,3.898973
std,1.074852,0.562376
min,0.000000,1.500000
25%,2.000000,3.500000
50%,3.000000,4.000000
75%,3.000000,4.333333
max,7.000000,5.000000


In [11]:
# Determinar Número de usuarios únicos en la tabla ratings y calificaciones promedio asignadas

query = """
SELECT
    username,
    COUNT(DISTINCT rating_id) AS "No. Calificaciones Asignadas"
FROM ratings
GROUP BY username
ORDER BY "No. Calificaciones Asignadas" DESC;
"""
result6 = pd.read_sql(query, engine)

d(m(f"### **Tabla Calificaciones por Usuario**"))
d(result6.head())

result6.describe()

### **Tabla Calificaciones por Usuario**

,username,No. Calificaciones Asignadas
0,martinadam,56
1,paul88,56
2,sfitzgerald,55
3,richard89,55
4,jennifermiller,53


,No. Calificaciones Asignadas
count,160.000000
mean,40.350000
std,5.607946
min,29.000000
25%,37.000000
50%,40.000000
75%,44.000000
max,56.000000


<div style="border-left: 10px solid #616161;padding: 20px;background: #D6D6D6;border-radius: 8px;color: #000000;word-wrap: break-word;overflow-wrap: break-word;white-space: normal;max-width: 98%;box-sizing: border-box;font-size: 15px;line-height: 1.6;text-align: justify;">

## **Conclusiones Parte 3.2. Calificaciones y reseñas por libro**

- El análisis de los **1,000 libros** muestra un promedio de **2.79 reseñas por libro**, con una mediana de **3** y un máximo de **7 reseñas**. Se identificaron además **6 libros sin reseñas**, lo que evidencia que la participación mediante reseñas no está distribuida de manera uniforme entre todo el catálogo.

- En cuanto a las calificaciones, los libros presentan un promedio de **3.90 puntos**, con una mediana de **4.00** y valores entre **1.50 y 5.00**. Esto indica que las valoraciones se concentran alrededor de 4 puntos.

- En cuanto a la actividad de los usuarios, se identificaron **160 usuarios únicos** de los cuales cada usuario realizó, en promedio, **40.35 calificaciones**, con un máximo de **56 calificaciones**. Se observa que un grupo reducido de usuarios presenta una actividad de calificación particularmente alta, llegando algunos a registrar más de 50 calificaciones.

- En conjunto, los resultados permiten identificar un catálogo amplio con niveles diferenciados de participación por libro y una actividad de calificación concentrada entre los usuarios registrados en la tabla `ratings`.

</div>

## 3.3 Desempeño de las editoriales según volumen de publicaciones relevantes
Determinar qué editorial ha publicado el mayor número de libros con **más de 50 páginas**.

- Relacionar las tablas `books` y `publishers`.
- Filtrar los libros con `num_pages > 50`.
- Agrupar los libros por editorial.
- Contabilizar el número de libros publicados por cada editorial.
- Ordenar los resultados para identificar la editorial con mayor cantidad de publicaciones.
- Obtener el resultado e interpretar el resultado en función del volumen de publicaciones.

In [12]:
# Editorial con mayor número de libros de más de 50 páginas

query = """
SELECT
    p.publisher_id,
    p.publisher,
    COUNT(b.book_id) AS "Total de Libros"
FROM books AS b
LEFT JOIN publishers AS p
    ON b.publisher_id = p.publisher_id
WHERE b.num_pages > 50
GROUP BY
    p.publisher_id,
    p.publisher
ORDER BY
    "Total de Libros" DESC;
"""

result3 = pd.read_sql(query, engine)

# Orden descendente por "Total de Libros"
result3 = result3.sort_values(
    by="Total de Libros", ascending=False).reset_index(drop=True)

d(m("## Libros publicados por editorial con más de 50 páginas"))
d(result3.head().style.hide(axis="index"))

result3.describe()

## Libros publicados por editorial con más de 50 páginas

publisher_id,publisher,Total de Libros
212,Penguin Books,42
309,Vintage,31
116,Grand Central Publishing,25
217,Penguin Classics,24
33,Ballantine Books,19


,publisher_id,Total de Libros
count,334.000000,334.000000
mean,170.979042,2.970060
std,98.185992,4.377015
min,1.000000,1.000000
25%,86.250000,1.000000
50%,171.500000,1.000000
75%,254.750000,3.000000
max,340.000000,42.000000


<div style="border-left: 10px solid #616161;padding: 20px;background: #D6D6D6;border-radius: 8px;color: #000000;word-wrap: break-word;overflow-wrap: break-word;white-space: normal;max-width: 98%;box-sizing: border-box;font-size: 15px;line-height: 1.6;text-align: justify;">

## Conclusiones Parte 3.3 Desempeño de las editoriales según volumen de publicaciones relevantes

- Al analizar los libros con más de 50 páginas, se identificaron **334 editoriales** con publicaciones que cumplen este criterio. El promedio es de **2.97 libros por editorial**, mientras que la mediana es de **1 libro**, lo que indica que la mayoría de las editoriales presenta un volumen reducido de publicaciones dentro del catálogo analizado.

- **Penguin Books** concentra el mayor número de publicaciones, con **42 libros**, seguida por **Vintage**, con 31, y **Grand Central Publishing**, con 25. Por lo tanto, Penguin Books es la editorial con mayor volumen de publicaciones relevantes según el criterio establecido.

</div>

## 3.4 Identificación del autor con mayor calificación promedio
Determinar qué autor tiene la calificación promedio más alta considerando únicamente los libros que cuentan con **al menos 50 calificaciones**.
- Relacionar las tablas `books`, `authors` y `ratings`.
- Agrupar las calificaciones por libro.
- Contabilizar el número de calificaciones recibidas por cada libro.
- Filtrar los libros que cumplan el mínimo de 50 calificaciones.
- Calcular la calificación promedio de cada libro.
- Relacionar los resultados con sus respectivos autores.
- Comparar las calificaciones promedio para identificar al autor con el valor más alto.
- Obtener el resultado e interpretar el resultado considerando el criterio mínimo de participación establecido.

In [13]:
# Autor con mayor calificación promedio entre libros con al menos 50 calificaciones

query = """SELECT
    a.author_id,
    a.author,
    AVG(r.rating) AS calificacion_promedio
    FROM
    ratings AS r
    JOIN books AS b ON r.book_id = b.book_id
    JOIN authors AS a ON b.author_id = a.author_id
    WHERE
    r.book_id IN (
    SELECT
    book_id
    FROM
    ratings
    GROUP BY
    book_id
    HAVING
    COUNT(*) >= 50
    )
    GROUP BY
    a.author_id,
    a.author
    ORDER BY
    calificacion_promedio DESC;"""

result4 = pd.read_sql(query, engine)

result4 = result4.sort_values(
    by="calificacion_promedio", ascending=False).reset_index(drop=True)


d(m("## Calificación promedio por autor en libros con al menos 50 calificaciones"))
d(result4.head())

result4.describe()

## Calificación promedio por autor en libros con al menos 50 calificaciones

,author_id,author,calificacion_promedio
0,236,J.K. Rowling/Mary GrandPré,4.287097
1,402,Markus Zusak/Cao Xuân Việt Khương,4.264151
2,240,J.R.R. Tolkien,4.246914
3,376,Louisa May Alcott,4.192308
4,498,Rick Riordan,4.080645


,author_id,calificacion_promedio
count,14.000000,14.000000
mean,374.642857,3.920135
std,163.680555,0.241052
min,106.000000,3.622951
25%,237.000000,3.743444
50%,374.000000,3.807528
75%,490.750000,4.164392
max,630.000000,4.287097


<div style="border-left: 10px solid #616161;padding: 20px;background: #D6D6D6;border-radius: 8px;color: #000000;word-wrap: break-word;overflow-wrap: break-word;white-space: normal;max-width: 98%;box-sizing: border-box;font-size: 15px;line-height: 1.6;text-align: justify;">

## Conclusiones Parte 3.4 Identificación del autor con mayor calificación promedio

- Al considerar únicamente los libros con al menos **50 calificaciones**, se identificaron **14 autores** que cumplen con el criterio establecido. <br>
La calificación promedio entre estos autores fue de **3.92**, con valores que van de **3.62 a 4.29**.

- El autor con la calificación promedio más alta fue **J.K. Rowling/Mary GrandPré**, con **4.29**. <br>
   Seguido por **Markus Zusak/Cao Xuân Việt Khương** con **4.26** y **J.R.R. Tolkien** con **4.25**. 

   </div>


## 3.5 Actividad de los usuarios según volumen de calificaciones

Determinar el número promedio de reseñas de texto entre los usuarios que calificaron **más de 50 libros**.

- Identificar los usuarios que hayan calificado más de 50 libros.
- Contabilizar el número de libros calificados por cada usuario.
- Filtrar los usuarios que cumplan el criterio de más de 50 calificaciones.
- Calcular el número de reseñas de texto realizadas por cada usuario.
- Calcular el promedio de reseñas de texto entre los usuarios seleccionados.
- Obtener el resultado e interpretar el nivel de participación de este grupo de usuarios.

In [14]:
# Promedio de reseñas de texto entre usuarios que calificaron más de 50 libros

query = """
SELECT
    -- Promedio de reseñas por usuario
    AVG(total_reseñas) AS "Promedio de reseñas de texto"
FROM (
    SELECT
        -- Identificar al usuario
        r.username,

        -- Contar sus reseñas
        COUNT(DISTINCT rv.review_id) AS total_reseñas

    FROM ratings AS r

    -- Conservar usuarios aunque no tengan reseñas
    LEFT JOIN reviews AS rv
        ON r.username = rv.username

    -- Seleccionar usuarios con más de 50 libros calificados
    WHERE r.username IN (
        SELECT username
        FROM ratings
        GROUP BY username
        HAVING COUNT(DISTINCT book_id) > 50
    )

    -- Agrupar las reseñas por usuario
    GROUP BY
        r.username
) AS reseñas_usuario;
"""

result5 = pd.read_sql(query, engine)

d(m("## Promedio de reseñas de texto entre usuarios con más de 50 calificaciones"))
d(result5)

## Promedio de reseñas de texto entre usuarios con más de 50 calificaciones

,Promedio de reseñas de texto
0,24.333333


<div style="border-left: 10px solid #616161;padding: 20px;background: #D6D6D6;border-radius: 8px;color: #000000;word-wrap: break-word;overflow-wrap: break-word;white-space: normal;max-width: 98%;box-sizing: border-box;font-size: 15px;line-height: 1.6;text-align: justify;">

## Conclusiones Parte 3.5 Actividad de los usuarios según volumen de calificaciones

- Entre los usuarios que calificaron más de **50 libros**, se obtuvo un promedio de **24.33 reseñas de texto por usuario**. Este resultado indica que, dentro de este grupo de usuarios con alta actividad de calificación, existe también una participación considerable mediante reseñas escritas.

</div>


<div style="border-left: 10px solid #2196F3;padding: 20px;background: #D6D6D6;border-radius: 8px;color: #000000;word-wrap: break-word;overflow-wrap: break-word;white-space: normal;max-width: 98%;box-sizing: border-box;font-size: 15px;line-height: 1.6;text-align: justify;">


## Conclusiones Parte 3. Análisis Exploratorio de Datos (EDA)

- El análisis exploratorio permitió caracterizar el catálogo de libros y la participación de los usuarios a partir de los cinco indicadores establecidos. De los **1,000 libros** registrados, **819 fueron publicados después del 1 de enero de 2000**, representando el **81.9 % del catálogo**.

- En cuanto a la participación de los usuarios, los libros presentan un promedio de **2.79 reseñas**, mientras que la calificación promedio es de **3.90**. Se identificaron además **6 libros sin reseñas**, lo que muestra que la participación no es uniforme entre los títulos analizados.

- Al aplicar el criterio de más de **50 páginas**, se identificaron **334 editoriales** con publicaciones que cumplen esta condición. **Penguin Books** presentó el mayor volumen, con **42 libros**, siendo la editorial con mayor número de publicaciones relevantes dentro del catálogo.

- Respecto a los autores, al considerar únicamente los libros con al menos **50 calificaciones**, se identificaron **14 autores** que cumplen el criterio. **J.K. Rowling/Mary GrandPré** obtuvo la calificación promedio más alta, con **4.29**.

- Finalmente, entre los usuarios que calificaron más de **50 libros**, el promedio fue de **24.33 reseñas de texto por usuario**, lo que evidencia que este grupo de usuarios presenta una participación significativa mediante reseñas escritas.

- En conjunto, los resultados muestran un catálogo compuesto principalmente por publicaciones posteriores al año 2000, con diferencias en el nivel de participación y valoración de los usuarios, así como una concentración del volumen de publicaciones entre determinadas editoriales y una mayor valoración promedio en algunos autores bajo los criterios establecidos.

</div>

<div style="border-left: 10px solid #4CAF50;padding: 20px;background: #D6D6D6;border-radius: 8px;color: #000000;word-wrap: break-word;overflow-wrap: break-word;white-space: normal;max-width: 98%;box-sizing: border-box;font-size: 15px;line-height: 1.6;text-align: justify;">

## **Conclusiones Finales**

- El análisis permitió caracterizar el catálogo de **1,000 libros** desde diferentes perspectivas, identificando que **819 libros (81.9 %)** fueron publicados después del **1 de enero de 2000**. Esto muestra que el catálogo está compuesto principalmente por publicaciones posteriores a este periodo.

- La interacción de los usuarios presenta diferentes niveles de participación. Los libros registran un promedio de 2.79 reseñas por título y una calificación promedio de 3.90, mientras que en la tabla ratings se identificaron 160 usuarios únicos, con un promedio de 40.35 calificaciones por usuario y un máximo de 56 calificaciones. Se observa que un grupo reducido de usuarios presenta una actividad de calificación particularmente alta, llegando algunos a registrar más de 50 calificaciones.

- El análisis de las editoriales permitió identificar a **Penguin Books** como la editorial con mayor número de publicaciones de más de **50 páginas**, con **42 libros**. Por otra parte, al aplicar el criterio de al menos **50 calificaciones**, **J.K. Rowling/Mary GrandPré** presentó la calificación promedio más alta, con **4.29 puntos**. Estos resultados permiten identificar referentes destacados tanto en volumen de publicaciones como en valoración de los usuarios bajo los criterios establecidos.

- Entre los usuarios con mayor actividad de calificación, definidos como aquellos que calificaron más de **50 libros**, se obtuvo un promedio de **24.33 reseñas de texto por usuario**. Al compararlo con el volumen de calificaciones utilizado para seleccionar este grupo, se observa que la participación mediante reseñas escritas es menos frecuente que la actividad de calificación, incluso entre los usuarios más activos.

- En conjunto, el análisis proporciona una visión integral del catálogo y de la interacción de los usuarios con la plataforma. Los resultados pueden servir como base para orientar estrategias de desarrollo del producto, especialmente en aspectos relacionados con la **amplitud del catálogo, la participación de los usuarios, la generación de reseñas y la identificación de libros, autores y editoriales con mayor nivel de interacción o relevancia dentro de los criterios analizados**.

</div>